In [5]:
import tiktoken
import torch
tokenizer = tiktoken.get_encoding("gpt2")

# Load dataset
file_path = "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

print("# chars: ", len(text_data))
print("# tokens: ", len(tokenizer.encode(text_data)))

# chars:  20479
# tokens:  5145


In [6]:
train_ratio = 0.8
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

print(split_idx)
print(len(val_data))
print(len(text_data))


16383
4096
20479


In [7]:
from llms_from_scratch_utils import create_dataloader, GPT_CONFIG_124M

train_loader = create_dataloader(train_data, batch_size=2, 
                                 max_length=GPT_CONFIG_124M["context_len"],
                                 stride=GPT_CONFIG_124M["context_len"],
                                 drop_last=True,
                                 shuffle=True,
                                 num_workers=0)
val_loader = create_dataloader(val_data, batch_size=2, 
                               max_length=GPT_CONFIG_124M["context_len"],
                               stride=GPT_CONFIG_124M["context_len"],
                               drop_last=False, shuffle=False, num_workers=0)

In [8]:
def calc_loss_batch(input_batch, target_batch, model):
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0,1), 
                                             target_batch.flatten())
    return loss

def calc_loss_loader(model, data_loader, num_batches = None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches




### Training Loop

In [9]:
from llms_from_scratch_utils import generate_text_greedy

def train_model_simple(model, train_loader, val_loader,
                       optimizer, tokenizer, num_epochs, eval_freq, eval_num_batches,
                       start_context):
    train_losses, val_losses, track_tokens_seen = [],[],[]
    tokens_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader,
                                                      val_loader, eval_num_batches)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}"
                      f"Val loss {val_loss:.3f}")
        generate_and_print_sample(model, tokenizer, start_context)
    return train_losses, val_losses, track_tokens_seen

def evaluate_model(model, train_loader, val_loader, eval_num_batches):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(model, train_loader, eval_num_batches)
        val_loss = calc_loss_loader(model, val_loader, eval_num_batches)
    model.train()
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = torch.tensor(tokenizer.encode(
        start_context, allowed_special={'<|endoftext|>'})).unsqueeze(0)
    with torch.no_grad():
        tokens_ids = generate_text_greedy(model, encoded, 
                                          max_new_tokens=50, context_size=context_size)

    decoded_text = tokenizer.decode(tokens_ids.squeeze(0).tolist())
    print(decoded_text.replace("\n", " "))
    model.train()


In [10]:
from llms_from_scratch_core import GPTModel
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
optimizer = torch.optim.AdamW(model.parameters(), 
                              lr=0.0004, weight_decay=0.1)
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, tokenizer, num_epochs=2,
    eval_freq=5, eval_num_batches=5, 
    start_context="Every effort moves you,")


Ep 1 (Step 000000): Train loss 9.580Val loss 10.013
Every effort moves you, the, the, the, the, the,,, the, the.                                  
Every effort moves you, the                                                 


### Generation Techniques

Temperature scaling is just a fancy description for dividing the logits by a number greater than zero.

Top Gear sampling when combined with probabilistic sampling and temperature scaling can improve the text generation results. In top-k sampling, we can restrict the sample tokens to the top K most likely tokens and exclude all other tokens from the selection process by masking their probabilities scores.


In [21]:
def generate(model, start_context, max_new_tokens, context_size,
             temperature=0., top_k=None, eos_id=None):
    idx = torch.tensor(tokenizer.encode(
        start_context, allowed_special={'<|endoftext|>'})).unsqueeze(0)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float('-inf')),
                logits
            )
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            # samples from distribution
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            # just do greedy sampling
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_next == eos_id:
            break
        idx = torch.cat((idx, idx_next), dim=1)
    decoded_text = tokenizer.decode(idx.squeeze(0).tolist())
    return decoded_text

In [22]:
print(generate(model, "Every effort moves you", max_new_tokens=15, context_size=GPT_CONFIG_124M["context_len"], top_k=25, temperature=1.4))


generate_and_print_sample(model, tokenizer, "Every effort moves you")


Every effort moves you.














Every effort moves you, the                                                


## Saving and loading Pytorch models

In [18]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    }, 
    "model_and_optimizer.pth"
)
checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)

model_loaded = GPTModel(GPT_CONFIG_124M)
model_loaded.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model_loaded.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model_loaded.eval()

generate_and_print_sample(model_loaded, tokenizer, "Every effort moves you")


Every effort moves you, the                                                
